In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
import warnings
from sklearn.exceptions import ConvergenceWarning
import scipy.stats as stats
import matplotlib.pyplot as plt
from sklearn.linear_model import LassoCV, ElasticNetCV
from sklearn.linear_model import Lasso

# Model Fitting

In [ ]:
# download orginal dataset
df=pd.read_csv("case1Data.csv")

# create vector for target variable (y)
y_class=df["y"]

# create dataframe for features, numerical and categorical (x)
data=df.drop("y",axis=1)

# only categorical features
categorical_data=data[["C_01", "C_02", "C_03", "C_04", "C_05"]]

# only numeric features
numerical_data=data.drop(columns=categorical_data.columns)

numerical_data = numerical_data.fillna(numerical_data.median(numeric_only=True))
print(numerical_data.isna().sum())

categorical_data.drop(columns="C_02", inplace=True)

categorical_data = categorical_data.fillna("missing")
print(categorical_data.isna().sum())

categorical_data = pd.get_dummies(categorical_data)
categorical_data = categorical_data.astype(int)
categorical_data.head()

categorical_data.columns = categorical_data.columns.str.replace(".0", "", regex=False)
categorical_data.head()

X = pd.concat([numerical_data, categorical_data], axis=1)
print(X.shape)
X.head()

In [ ]:
def centerData(data):
    mu = np.mean(data,axis=0)
    data = data - mu
    return data, mu

In [ ]:
scaler = StandardScaler()

num_cols = numerical_data.columns
num_idx = [X.columns.get_loc(c) for c in num_cols]
X_scaled = X.values.copy()
X_scaled[:, num_idx] = scaler.fit_transform(X_scaled[:, num_idx])

y_centered, mu = centerData(y_class.values)

# select best alpha
lasso_final = LassoCV(cv=5, fit_intercept=False)
lasso_final.fit(X_scaled, y_centered)
best_alpha = lasso_final.alpha_
print(f"Selected alpha: {best_alpha}")

# see how many coefs are not zero
coef_final = lasso_final.coef_ 
non_zero_count = np.sum(coef_final != 0)
print(f"Number of non-zero coefficients: {non_zero_count}")

In [ ]:
mse_path = lasso_final.mse_path_ 
mse_mean = np.mean(mse_path, axis=1)
mse_std = np.std(mse_path, axis=1) / np.sqrt(mse_path.shape[1])

min_idx = np.argmin(mse_mean)
min_mse = mse_mean[min_idx]
min_se = mse_std[min_idx]

threshold = min_mse + min_se

# find largest alpha with MSE <= threshold
alphas = lasso_final.alphas_
one_se_idx = np.where(mse_mean <= threshold)[0] 
alpha_1se = np.max(alphas[one_se_idx]) 

# coefs of 1se alpha
# refit on the full scaled data, coefs are not stored by lassolars cv
lasso_1se = Lasso(alpha=alpha_1se, fit_intercept=False)
lasso_1se.fit(X_scaled, y_centered)
coef_1se = lasso_1se.coef_
non_zero_1se = np.sum(coef_1se != 0)

print(f"Alpha chosen by 1 SE rule: {alpha_1se}")
print(f"Number of non-zero coefficients (1 SE): {non_zero_1se}")
print("\n")
print(f"Alpha chosen by min CV MSE: {best_alpha}")
print(f"Number of non-zero coefficients (min CV MSE): {non_zero_count}")


plt.figure(figsize=(12,6))
plt.errorbar(alphas, mse_mean, yerr=mse_std, fmt='-o', ecolor='lightgray', capsize=3,label='Mean CV MSE ± SE')
plt.axvline(best_alpha, color='red', linestyle='--',label=f'Min CV MSE alpha = {best_alpha:.4f}')
plt.axvline(alpha_1se, color='green', linestyle='--',label=f'1-SE alpha = {alpha_1se:.4f}')
plt.xlabel("Alpha")
plt.ylabel("Mean CV MSE")
plt.title("LASSO - coordinate descent CV Error vs Alpha")
plt.xscale('log')
plt.gca().invert_xaxis()
plt.legend()
plt.show()

# Regression on new data

In [ ]:
X_new = pd.read_csv("case1Data_Xnew.csv")

categorical_new = X_new[["C_01", "C_02", "C_03", "C_04", "C_05"]]
numerical_new = X_new.drop(columns=categorical_new.columns)

categorical_new = categorical_new.drop(columns="C_02")
categorical_new = categorical_new.fillna("missing")

categorical_new = pd.get_dummies(categorical_new)
categorical_new.columns = categorical_new.columns.str.replace(".0", "", regex=False)

categorical_new = categorical_new.reindex(columns=categorical_data.columns, fill_value=0)

numerical_new = numerical_new.fillna(numerical_data.median())

X_new_processed = pd.concat([numerical_new, categorical_new], axis=1)

In [ ]:
X_new_scaled = X_new_processed.values.copy()
X_new_scaled[:, num_idx] = scaler.transform(X_new_scaled[:, num_idx])

In [ ]:
y_pred_centered = X_new_scaled @ coef_1se

In [ ]:
y_pred = y_pred_centered + mu

In [ ]:
pd.DataFrame(y_pred, columns=["y"]).to_csv("predictions.csv", index=False)